<a href="https://colab.research.google.com/github/Rashmi000Rashmi/Data-structures-/blob/main/Copy_of_CommitHunter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# Clone the OpenJ9 repository and go inside it
!git clone https://github.com/eclipse-openj9/openj9.git
%cd openj9

Cloning into 'openj9'...
remote: Enumerating objects: 291787, done.
remote: Counting objects: 100% (828/828), done.
remote: Compressing objects: 100% (496/496), done.
remote: Total 291787 (delta 623), reused 332 (delta 332), pack-reused 290959 (from 4)
Receiving objects: 100% (291787/291787), 192.70 MiB | 21.77 MiB/s, done.
Resolving deltas: 100% (221004/221004), done.
Updating files: 100% (10334/10334), done.
/content/openj9/openj9


In [9]:
# 🔁 Replace with your known SHAs
GOOD_SHA = "ab9584ee40"
BAD_SHA = "4d92969242"

print("✅ Good SHA:", GOOD_SHA)
print("❌ Bad SHA:", BAD_SHA)


✅ Good SHA: ab9584ee40
❌ Bad SHA: 4d92969242


In [10]:
# 🔍 Get commit messages and changed files between GOOD and BAD SHAs
log_output = !git log --oneline {GOOD_SHA}..{BAD_SHA}
diff_output = !git diff --name-status {GOOD_SHA} {BAD_SHA}

# ✅ Show results
print("=== Commits ===")
for line in log_output:
    print(line)

print("\n=== Changed Files ===")
for line in diff_output:
    print(line)


=== Commits ===
4d92969242 Merge pull request #21856 from luke-li-2003/vecHashOffeHeap
cf09cea809 Merge pull request #21802 from AdamBrousseau/bootjdk_cleanup_job
2d480d1dec Merge pull request #21866 from mpirvu/replay-cleanout
d8c2445a0f Merge pull request #21863 from pshipton/nasm
4d87ccc745 Add jenkins script for bootjdk cleanup
8aa5ac94f9 Delete unused IProfiler code related to comp replay
73ad1e4c65 Update minimum nasm dependency from 2.13.03 to 2.15.05
196082df05 Merge pull request #21857 from dmitripivkine/master
97fd30b19c Do not add Own.Sync. object to the list if scan wasn't successful
a85b56b1ff Enable offheap support in codegen
58c12750db Merge pull request #21851 from mpirvu/setcallcount-cleanup
eea7a9b0f1 Enable vectorizedHashCode intrinsic for OffHeap on POWER
07954346a0 Delete isWarmCallGraphCall() and other unused related methods
b0fe5ec788 Merge pull request #21828 from fengxue-IS/21037
43b64dd24c Merge pull request #21819 from nbhuiyan/ecs-forceinline-final
3dc11e735

In [11]:
# 🔄 Recollect commit logs and file diffs
log_output = !git log --oneline {GOOD_SHA}..{BAD_SHA}
diff_output = !git diff --name-status {GOOD_SHA} {BAD_SHA}

In [12]:
import subprocess

# ✅ Use the SHAs you've already set
output_file = "gpt_input_prompt.txt"

with open(output_file, "w") as f:
    f.write("==== GPT INPUT FOR COMMIT HUNTER ====\n\n")
    f.write(f"Good Build SHA: {GOOD_SHA}\n")
    f.write(f"Bad Build SHA:  {BAD_SHA}\n\n")

    # Commits between SHAs
    f.write("=== Commits Between Builds ===\n")
    commits = subprocess.check_output(["git", "log", "--oneline", f"{GOOD_SHA}..{BAD_SHA}"]).decode()
    f.write(commits + "\n")

    # Commit messages with file names
    f.write("=== Commit Messages & Changed Files ===\n")
    log_cmd = ["git", "log", f"{GOOD_SHA}..{BAD_SHA}", "--pretty=format:Commit: %h | %an | %s%n", "--name-only"]
    commit_details = subprocess.check_output(log_cmd).decode()
    f.write(commit_details + "\n")

    # Changed files
    f.write("=== Changed Files (Name Status) ===\n")
    changed_files = subprocess.check_output(["git", "diff", "--name-status", GOOD_SHA, BAD_SHA]).decode()
    f.write(changed_files + "\n")

    # Full diff (optional)
    f.write("=== Full Patch/Diff ===\n")
    full_diff = subprocess.check_output(["git", "diff", GOOD_SHA, BAD_SHA]).decode()
    f.write(full_diff + "\n")

    # Test failure placeholder
    f.write("=== Failed Test Context (Fill this part manually) ===\n")
    f.write("Failed Test Name: _______________________\n")
    f.write("Stack Trace: ____________________________\n\n")

    # Final GPT prompt
    f.write("""=== ChatGPT Prompt ===
Based on the commits, changed files, and the failed test/stack trace,
which commit is most likely responsible for the build failure?

Explain your reasoning in detail.
""")

print("✅ File 'gpt_input_prompt.txt' created successfully!")


✅ File 'gpt_input_prompt.txt' created successfully!


In [16]:
import subprocess

output_file = "gpt_input_prompt.txt"

with open(output_file, "w") as f:
    f.write("==== GPT INPUT FOR COMMIT HUNTER (SHORT VERSION) ====\n\n")
    f.write(f"Good Build SHA: {GOOD_SHA}\n")
    f.write(f"Bad Build SHA:  {BAD_SHA}\n\n")

    # Commits
    f.write("=== Commit Messages (Condensed) ===\n")
    commits = subprocess.check_output(["git", "log", "--pretty=format:%h | %an | %s", f"{GOOD_SHA}..{BAD_SHA}"]).decode()
    f.write(commits[:3000] + "\n\n")  # Limit to ~3000 characters

    # Changed files
    f.write("=== Changed Files (Name Status) ===\n")
    changed_files = subprocess.check_output(["git", "diff", "--name-status", GOOD_SHA, BAD_SHA]).decode()
    f.write(changed_files[:1000] + "\n\n")  # Limit to ~1000 characters

    # Placeholder for test failure
    f.write("=== Failed Test Context (Fill this part manually) ===\n")
    f.write("Failed Test Name: testLoginFailure\n")
    f.write("Stack Trace:\njava.lang.AssertionError: Login failed for valid credentials\n\tat LoginService.java:42\n\n")

    # Final GPT prompt
    f.write("""=== ChatGPT Prompt ===
Based on the commits, changed files, and the test failure and stack trace,
which commit is most likely responsible for the regression?

Give your reasoning clearly and rank the top 1–2 suspects.
""")

print("✅ Trimmed 'gpt_input_prompt.txt' generated.")


✅ Trimmed 'gpt_input_prompt.txt' generated.


In [13]:
!pip install openai


In [ ]:
!pip uninstall -y openai
!pip install --upgrade openai

In [17]:
import openai

# 🔐 Set your key using the new client method
client = openai.OpenAI(api_key="GPT_KEY")  # replace with your key

# 📥 Load prompt from file
with open("gpt_input_prompt.txt", "r") as f:
    gpt_prompt = f.read()

# 🤖 Use the new v1 SDK Chat API
chat_response = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": "You are a software debugging assistant."},
        {"role": "user", "content": gpt_prompt}
    ],
    temperature=0.3,
    max_tokens=1000
)gt6

# 🖨️ Print response
print("📌 GPT Response:\n")
print(chat_response.choices[0].message.content)


📌 GPT Response:

The test failure is related to a login failure for valid credentials, which suggests that the issue is likely related to changes in authentication, security, or user management code. However, looking at the commit messages and the files changed, none of them seem to directly relate to such functionalities.

The commit "a85b56b1ff | Luke Li | Enable offheap support in codegen" could potentially have caused the issue if the login service relies on some specific behavior of the codegen that was affected by this change. But this is a less likely scenario.

The commit "97fd30b19c | Dmitri Pivkine | Do not add Own.Sync. object to the list if scan wasn't successful" could also be a potential cause if the login service uses synchronization objects and the change affected this. But again, this is a less likely scenario.

Without more information about the nature of the login service and how it interacts with the rest of the system, it's difficult to definitively pinpoint the ca